# V-ChartText – Cải thiện 4: prompt trích xuất v2 cho Qwen3-VL-2B (Kaggle)

Chỉ chạy lại **bước 1 (extract)** với prompt `v2` (`src/prompts.py`, `EXTRACT_TABLE_V2`), giữ nguyên mọi cấu hình khác của v1
(greedy, `max_new_tokens = 2048`, `max_side = 1280`, fp16 trên T4) để so sánh công bằng v1 ↔ v2.

Prompt v2 = prompt v1 + 5 quy tắc: giữ tiền tố đơn vị (nghìn/triệu/tỉ), không tự đặt tiêu đề, chép nguyên văn tên chuỗi/lát,
quy ước series/x cho biểu đồ tròn, không đảo series/x ở biểu đồ miền.

**Cài đặt notebook** (Settings bên phải): *Accelerator = GPU T4 x2*, *Internet = On*,
*Add-ons → Secrets*: `GH_TOKEN` (GitHub token đọc được repo `rangha26/V-ChartText`), *Input*: dataset ảnh biểu đồ của nhóm.

Thời gian: ~200 ảnh × ~19 s chia 2 GPU ≈ 35–40 phút. Bấm **Save Version → Save & Run All (Commit)** để chạy nền.
Bị ngắt giữa chừng: tạo phiên bản mới, *Add Input → Notebook Output* của phiên bản cũ, chạy lại; cell 2 chép phần đã chạy vào và script bỏ qua ảnh đã có.

In [ ]:
# 1. Cấu hình
GH_USER = 'rangha26'
BRANCH  = 'dev'
REPO    = '/tmp/V-ChartText'
OUT     = '/kaggle/working/outputs'
PARTS   = '/kaggle/working/parts'
QWEN    = 'Qwen/Qwen3-VL-2B-Instruct'
PV      = 'v2'                       # phiên bản prompt
TEST_ONLY = False                    # True: chỉ chạy thử 6 ảnh rồi dừng (kiểm tra prompt trước khi chạy hết)

import glob, os, unicodedata, shutil, json, re, statistics
nfc = lambda s: unicodedata.normalize('NFC', s).lower()
RAW = next(d for d in sorted(glob.glob('/kaggle/input/**/', recursive=True))
           if any('cột' in nfc(x) for x in os.listdir(d)))
print('RAW =', RAW)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 2. Lấy code + khôi phục phần đã chạy của phiên bản trước (nếu có)
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GH_TOKEN')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://{token}@github.com/{GH_USER}/V-ChartText.git {REPO}
%cd {REPO}
!git log --oneline -1

os.makedirs(OUT, exist_ok=True); os.makedirs(PARTS, exist_ok=True)
# outputs v1 có sẵn trong repo (để so sánh v1 vs v2) -> chép sang OUT rồi trỏ outputs/ vào OUT
if not os.path.islink('outputs'):
    for f in glob.glob('outputs/*.jsonl'):
        shutil.copy(f, OUT)
    shutil.copytree('outputs/pipeline', f'{OUT}/pipeline', dirs_exist_ok=True)
    shutil.rmtree('outputs'); os.symlink(OUT, 'outputs')
for f in glob.glob('/kaggle/input/**/parts/*.jsonl', recursive=True):
    dst = os.path.join(PARTS, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst); print('khôi phục', dst)
print(sorted(os.listdir(OUT)))

In [ ]:
# 3. Thư viện (Qwen3-VL cần transformers >= 4.57)
!pip install -q "transformers>=4.57.0" accelerate
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)"

In [ ]:
# 4. Ảnh: chép từ dataset vào data/images theo manifest (không đụng tới nhãn)
!python src/prepare_dataset.py --raw "{RAW}" | tail -3
!python eval/validate_labels.py | tail -4
!python -c "import sys; sys.path.insert(0,'src'); from prompts import PROMPTS_BY_VERSION as P; print(P['{PV}']['extract'][-1200:])"

In [ ]:
# 5. Chạy thử 6 ảnh hay mắc lỗi ở v1 (mất 'Nghìn', tự đặt tiêu đề, tên lát, miền đảo series/x)
TEST_IDS = 'bar_001,bar_004,bar_040,bar_074,pie_011,area_008'
!python src/infer_qwen.py --model {QWEN} --task extract --prompt_version {PV} --ids {TEST_IDS} \
    --out /kaggle/working/test_{PV}.jsonl 2>&1 | grep -v Warning | tail -8
for l in open(f'/kaggle/working/test_{PV}.jsonl', encoding='utf-8'):
    r = json.loads(l); t = r['output']
    m = re.search(r'\{.*\}', re.sub(r'```(?:json)?', '', t), re.S)
    try:
        j = json.loads(m.group(0))
        print(f"{r['id']}: title={j.get('title')!r} | y_unit={(j.get('y_axis') or {}).get('unit')!r} | "
              f"{len(j.get('data') or [])} điểm | ví dụ: {(j.get('data') or [None])[0]}")
    except Exception:
        print(r['id'], 'JSON lỗi:', t[:200])
if TEST_ONLY:
    raise SystemExit('TEST_ONLY = True: dừng ở đây. Xem kết quả trên, đặt TEST_ONLY = False rồi chạy lại.')

In [ ]:
# 6. Chạy toàn bộ: chia đôi danh sách ảnh cho 2 GPU (mỗi GPU ghi một file phần, chạy tiếp được khi bị ngắt)
CMD = f'python src/infer_qwen.py --model {QWEN} --task extract --prompt_version {PV}'
!CUDA_VISIBLE_DEVICES=0 {CMD} --shard 0/2 --out {PARTS}/part0.jsonl > /kaggle/working/log_part0.txt 2>&1 & \
 CUDA_VISIBLE_DEVICES=1 {CMD} --shard 1/2 --out {PARTS}/part1.jsonl > /kaggle/working/log_part1.txt 2>&1 & wait
!tail -n 2 /kaggle/working/log_part*.txt

# gộp 2 phần -> outputs/Qwen3-VL-2B-Instruct__extract__v2.jsonl
final = f"{OUT}/{QWEN.split('/')[-1]}__extract__{PV}.jsonl"
seen, rows = set(), []
for f in sorted(glob.glob(f'{PARTS}/part*.jsonl')):
    for l in open(f, encoding='utf-8'):
        if l.strip() and (r := json.loads(l))['id'] not in seen:
            seen.add(r['id']); rows.append(r)
rows.sort(key=lambda r: r['id'])
with open(final, 'w', encoding='utf-8') as f:
    f.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in rows)
print(len(rows), 'ảnh ->', final)

In [ ]:
# 7. Kiểm tra nhanh v1 vs v2: số ảnh, JSON hợp lệ, chạm giới hạn token, thời gian
for f in sorted(glob.glob(f'{OUT}/Qwen*__extract__*.jsonl')):
    rs = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    ok = cut = 0
    for r in rs:
        cut += (r.get('n_new_tokens') or 0) >= (r.get('max_new_tokens') or 2048) - 5
        m = re.search(r'\{.*\}', re.sub(r'```(?:json)?', '', r['output']), re.S)
        try: json.loads(m.group(0)); ok += 1
        except Exception: pass
    print(f"{os.path.basename(f)}: {len(rs)} ảnh | JSON ok {ok} | chạm giới hạn {cut} | "
          f"{statistics.mean(r['latency_s'] for r in rs):.1f}s/ảnh")

In [ ]:
# 8. Chấm điểm v2, chạy pipeline trên v2, lập lại các bảng (v1 và v2 nằm cạnh nhau, v2 mang hậu tố @v2)
# eval/results không nằm trong git -> chấm lại cả các file v1 (im lặng) để bảng tổng hợp đủ cột
for f in sorted(glob.glob('outputs/*__extract__v1.jsonl')):
    !python eval/score_extract.py "{f}" > /dev/null
for f in sorted(glob.glob('outputs/*__describe__v1.jsonl')):
    !python eval/check_describe.py "{f}" > /dev/null
!python eval/score_extract.py outputs/Qwen3-VL-2B-Instruct__extract__{PV}.jsonl
!python src/pipeline/run.py --source pred --extract outputs/Qwen3-VL-2B-Instruct__extract__{PV}.jsonl
!python src/pipeline/run.py --source pred --extract outputs/Qwen3-VL-2B-Instruct__extract__{PV}.jsonl --uncertain hedge
!python eval/summarize.py   > eval/results/summary_models.md
!python eval/error_stats.py > eval/results/error_stats.md
!python eval/compare_pipeline.py > eval/results/compare_pipeline.md
!head -n 8 eval/results/summary_models.md | cut -c1-200
!grep -E "Qwen" eval/results/compare_pipeline.md | head -12

In [ ]:
# 9. Gom kết quả để tải về (Output -> Download): outputs/*__v2.jsonl, outputs/pipeline/*@v2*, results/
!mkdir -p /kaggle/working/results && cp eval/results/* /kaggle/working/results/ 2>/dev/null
!ls -la {OUT} {OUT}/pipeline /kaggle/working/results | head -60